# Week 2 — Day 3: MLflow setup + baseline run logging

Tasks:
- Set up MLflow locally; log the Day 2 baseline run (params, metrics)
- Explore light fine-tuning options: freezing the vision encoder and fine-tuning only the text
  decoder on a small Flickr8k training subset

Deliverable: first MLflow-tracked experiment run visible in the MLflow UI.

### 1. Point MLflow at a local file store

Tracking data is written to `../mlruns/` at the repo root (already gitignored — MLflow's own
storage isn't meant to live in git; the Model Registry entry gets formalized in Week 4).

In [1]:
import os
import mlflow
import pandas as pd

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("flickr8k-image-captioning")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: sqlite:///../mlflow.db


### 2. Load the Day 2 baseline results

In [2]:
scores_df = pd.read_csv('../data/processed/week2_baseline_eval_scores.csv')
predictions_df = pd.read_csv('../data/processed/week2_baseline_eval_predictions.csv')
scores = dict(zip(scores_df['metric'], scores_df['score']))
scores

{'BLEU (sacrebleu)': 16.65627009850249,
 'ROUGE-1': 50.19467205680772,
 'ROUGE-2': 26.64341957310328,
 'ROUGE-L': 48.08015151785763}

### 3. Log the zero-shot baseline as an MLflow run

In [3]:
with mlflow.start_run(run_name="zero-shot-blip-base") as run:
    mlflow.log_params({
        "model_name": "Salesforce/blip-image-captioning-base",
        "fine_tuned": False,
        "decoding_strategy": "greedy",
        "max_new_tokens": 30,
        "val_size": len(predictions_df),
        "device": "cpu",
    })
    mlflow.log_metrics({
        "bleu": scores["BLEU (sacrebleu)"],
        "rouge1": scores["ROUGE-1"],
        "rouge2": scores["ROUGE-2"],
        "rougeL": scores["ROUGE-L"],
    })
    mlflow.log_artifact('../data/processed/week2_baseline_eval_scores.csv')
    mlflow.log_artifact('../data/processed/week2_baseline_eval_predictions.csv')
    if os.path.exists('../data/processed/week2_zero_shot_blip_grid.png'):
        mlflow.log_artifact('../data/processed/week2_zero_shot_blip_grid.png')
    run_id = run.info.run_id

print(f"Logged run: {run_id}")

Logged run: 99867b61af684de89f0abc8a655df88f


### 4. Verify the run is tracked

In [4]:
client = mlflow.tracking.MlflowClient()
run_data = client.get_run(run_id)
print("Params:", run_data.data.params)
print("Metrics:", run_data.data.metrics)
print("\nTo view in the MLflow UI, run from the repo root:\n  .venv\\Scripts\\mlflow ui --backend-store-uri sqlite:///mlflow.db\nthen open http://127.0.0.1:5000")

Params: {'model_name': 'Salesforce/blip-image-captioning-base', 'fine_tuned': 'False', 'decoding_strategy': 'greedy', 'max_new_tokens': '30', 'val_size': '300', 'device': 'cpu'}
Metrics: {'bleu': 16.65627009850249, 'rouge1': 50.19467205680772, 'rouge2': 26.64341957310328, 'rougeL': 48.08015151785763}

To view in the MLflow UI, run from the repo root:
  .venv\Scripts\mlflow ui --backend-store-uri sqlite:///mlflow.db
then open http://127.0.0.1:5000


### 5. Fine-tuning plan for Day 3-4 (executed on Kaggle GPU)

Local CPU is too slow to fine-tune BLIP in reasonable time, so the actual fine-tuning runs happen
on **Kaggle GPU notebooks** (per project setup) — see `notebooks/kaggle_week2_finetune_blip.ipynb`.
Approach:

- **Freeze the vision encoder**, fine-tune only the text decoder — keeps training cheap and avoids
  overfitting the small subset, while still adapting caption style/vocabulary to Flickr8k
- **Training subset:** a few thousand image-caption pairs (not all 40k captions) to keep each Kaggle
  run short
- **Experiments to compare (2-3 runs):**
  1. Baseline fine-tune: default learning rate (e.g. 5e-5), greedy decoding
  2. Lower learning rate (e.g. 1e-5) — check for more stable convergence
  3. Beam search decoding at inference instead of greedy — no retraining needed, just a decoding
     strategy comparison
- Each Kaggle run logs its own params/metrics/checkpoint to its output; results are pulled back
  down and logged into this local MLflow store (Day 4) so all runs — zero-shot baseline included —
  are comparable in one place.